In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# LongSplat 환경 세팅 (Colab A100)

> **[NVlabs/LongSplat](https://github.com/NVlabs/LongSplat)** — ICCV 2025  

---

## ✅ 순서
1. GPU 확인
2. Git Clone
3. CUDA / GCC 세팅
4. PyTorch 및 패키지 설치
5. Submodule 빌드
6. 데이터셋 업로드
7. 학습 실행


## 0. GPU 확인

In [ ]:
!nvidia-smi

Wed Apr 15 10:28:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Git Clone

In [ ]:
!git clone --recursive https://github.com/NVlabs/LongSplat.git
%cd /content/LongSplat

Cloning into 'LongSplat'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 150 (delta 58), reused 59 (delta 46), pack-reused 62 (from 1)
Receiving objects: 100% (150/150), 135.07 MiB | 19.36 MiB/s, done.
Resolving deltas: 100% (60/60), done.
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/linjohnss/diff-gaussian-rasterization-w-pose.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/MrNeRF/optimized-fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/mast3r' (https://github.com/naver/mast3r.git) registered for path 'submodules/mast3r'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn) registered for path 'submodules/simple-knn'
Cloning into '/content/LongSplat/submodules/diff-gaussian-rasterization'...
remote: Enumerating objects: 222, don

## 2. CUDA 12.1 PATH 설정 및 GCC 9 설치

In [ ]:
import os
os.environ["PATH"] = "/usr/local/cuda-12.1/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-12.1/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

In [ ]:
!gcc --version

gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [ ]:
!apt-get install -y gcc-9 g++-9 -q
!update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-9 9
!update-alternatives --install /usr/bin/g++ g++ /usr/bin/g++-9 9
!update-alternatives --set gcc /usr/bin/gcc-9
!update-alternatives --set g++ /usr/bin/g++-9
!gcc --version

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  cpp-9 gcc-9-base libasan5 libgcc-9-dev libstdc++-9-dev
Suggested packages:
  gcc-9-locales g++-9-multilib gcc-9-doc gcc-9-multilib libstdc++-9-doc
The following NEW packages will be installed:
  cpp-9 g++-9 gcc-9 gcc-9-base libasan5 libgcc-9-dev libstdc++-9-dev
0 upgraded, 7 newly installed, 0 to remove and 42 not upgraded.
Need to get 41.4 MB of archives.
After this operation, 138 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 gcc-9-base amd64 9.5.0-1ubuntu1~22.04.1 [204 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 cpp-9 amd64 9.5.0-1ubuntu1~22.04.1 [10.6 MB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libasan5 amd64 9.5.0-1ubuntu1~22.04.1 [3,141 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libgcc-9-dev amd64 9.

## 3. PyTorch 2.5.1 + CUDA 12.1 설치

In [ ]:
!pip install torch==2.5.1 torchvision --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 110.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 59.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 143.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 48.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA L4


## 4. pytorch3d & torch_scatter 설치 (pre-built wheel)

In [ ]:
!pip install --extra-index-url https://miropsota.github.io/torch_packages_builder \
    pytorch3d==0.7.8+pt2.5.1cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!pip install torch_scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 11.9 MB/s eta 0:00:00


## 5. requirements.txt 설치

In [ ]:
!grep -v "pytorch3d" /content/LongSplat/requirements.txt > /tmp/req_no_pytorch3d.txt
!pip install -r /tmp/req_no_pytorch3d.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.8/157.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.0 MB/s eta 0:00:00


## 6. Submodule 빌드

In [ ]:
import re

path = "/content/LongSplat/submodules/fused-ssim/setup.py"
with open(path, "r") as f:
    content = f.read()

#print(content)  # 내용 확인, -arch=sm_89 부분을 A100에 맞게 수정

#content_fixed = content.replace("sm_89", "sm_80").replace("8.9", "8.0")

#with open(path, "w") as f:
#    f.write(content_fixed)

#print("수정 완료")
#print(open(path).read())

In [ ]:
# A100: Compute Capability 8.0 / L4: 8.9
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.9"

%cd submodules/fused-ssim
!pip install -e . --no-build-isolation
%cd /content/LongSplat

/content/LongSplat/submodules/fused-ssim
Obtaining file:///content/LongSplat/submodules/fused-ssim
  Preparing metadata (setup.py) ... done
  Running setup.py develop for fused_ssim
/content/LongSplat


In [ ]:
!pip install submodules/simple-knn --no-build-isolation

Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl size=2929409 sha256=8707af72670eeae087d91ce638667af7923c0b66791fe9ac74448e3589e751c7
  Stored in directory: /root/.cache/pip/wheels/21/ec/54/fc1ee6f1100aa2f7eeeca035227afdb82e07aa0705effa7dfe
Successfully built simple_knn


In [ ]:
!pip install submodules/diff-gaussian-rasterization --no-build-isolation

Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp312-cp312-linux_x86_64.whl size=3170366 sha256=82ac61ac20635b065fd74ad4bc141a5c2221ca3a1ededd48e1f121ccefa25e1d
  Stored in directory: /root/.cache/pip/wheels/5e/62/d6/b5c705867591fe463f76a3fb6da5a0fcce383149357d1974d5
Successfully built diff_gaussian_rasterization


In [ ]:
%cd /content/LongSplat

/content/LongSplat


## 7. (선택) RoPE CUDA 커널 컴파일

런타임 속도 향상을 위한 선택 사항입니다.

In [ ]:
%cd submodules/mast3r/dust3r/croco/models/curope/
!python setup.py build_ext --inplace
%cd /content/LongSplat

/content/LongSplat/submodules/mast3r/dust3r/croco/models/curope
running build_ext
/usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py:497: UserWarning: Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
  warnings.warn(msg.format('we could not find ninja.'))
/usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py:416: UserWarning: The detected CUDA version (12.8) has a minor version mismatch with the version that was used to compile PyTorch (12.1). Most likely this shouldn't be a problem.
  warnings.warn(CUDA_MISMATCH_WARN.format(cuda_str_version, torch.version.cuda))
/usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py:426: UserWarning: There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.8
  warnings.warn(f'There are no {compiler_name} version bounds defined for CUDA version {cuda_str_version}')
building 'curope' extension
creating bu

## 8. 학습 실행

파라미터를 수정한 뒤 셀을 다시 실행하면 스크립트가 덮어씌워지고 바로 학습이 시작됩니다.

In [21]:
import subprocess
result = subprocess.run(
    ["find", "/content/drive/MyDrive/Research/26_Capstone2/input", "-name", ".DS_Store", "-delete"],
    capture_output=False, text=True
)
print("삭제 완료")

삭제 완료


In [ ]:
import os, random, textwrap, subprocess, sys, subprocess
from datetime import datetime, timezone, timedelta
from pathlib import Path

from tqdm.notebook import tqdm
import tqdm as tqdm_module
tqdm_module.std.tqdm = tqdm

KST = timezone(timedelta(hours=9))

# ✏️ 처리할 scene 목록
# 'grass_rain',  'grass_snow',
# 'hydrant_rain','hydrant_snow',
# 'pillar_rain', 'pillar_snow',
# 'road_rain',   'road_snow',
# 'sky_rain',    'sky_snow',
# 'stair_rain',  'stair_snow',

SCENES = [
    'pillar_original',
    'pillar_snow'
]

for scene in SCENES:
    result = subprocess.run(
    ["find", "/content/drive/MyDrive/Research/26_Capstone2/input", "-name", ".DS_Store", "-delete"],
    capture_output=False, text=True
    )
    print("삭제 완료")

    input_path  = f'/content/drive/MyDrive/Research/26_Capstone2/input/{scene}'
    r           = 1
    pose_iter   = 100
    local_iter  = 200
    global_iter = 500
    port        = random.randint(10000, 30000)
    timestamp   = datetime.now(KST).strftime("%Y-%m-%d_%H:%M")
    outputs     = f"/content/drive/MyDrive/Research/26_Capstone2/LongSplat/{scene}/{timestamp}"

    script = textwrap.dedent(f"""
        #!/bin/bash
        ulimit -n 4096
        cd /content/LongSplat
        python train.py --eval -s {input_path} -m {outputs} --port {port} \\
            --images input --mode custom -r {r} \\
            --pose_iteration {pose_iter} --local_iter {local_iter} --global_iter {global_iter}
        python render.py  -m {outputs}
        python metrics.py -m {outputs}
    """).strip()

    script_path = Path("/content/LongSplat/scripts/train_custom.sh")
    script_path.write_text(script, encoding="utf-8")
    print(f"\n{'='*60}\nScene: {scene}\n{script}\n{'='*60}")

    with subprocess.Popen(
        ["bash", str(script_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"}
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)

    if proc.returncode != 0:
        raise RuntimeError(f"[{scene}] Script failed (exit code {proc.returncode})")

    print(f"\n[{scene}] ✓ 완료")

Streaming output truncated to the last 5000 lines.
Global Optimization 28/175: 100%|██████████| 500/500 [00:21<00:00, 23.04it/s, Loss=0.0376187]

Pose estiamtion progress: 100%|██████████| 100/100 [00:02<00:00, 39.64it/s, Loss=0.0238465]

Local Optimization 29/175(w=17): 100%|██████████| 200/200 [00:07<00:00, 25.48it/s, Loss=0.0375730]

Global Optimization 29/175: 100%|██████████| 500/500 [00:21<00:00, 23.37it/s, Loss=0.0395847]

Pose estiamtion progress: 100%|██████████| 100/100 [00:02<00:00, 41.61it/s, Loss=0.0185507]

Local Optimization 30/175(w=18): 100%|██████████| 200/200 [00:07<00:00, 25.57it/s, Loss=0.0358275]

Global Optimization 30/175: 100%|██████████| 500/500 [00:21<00:00, 23.50it/s, Loss=0.0411424]

Pose estiamtion progress: 100%|██████████| 100/100 [00:02<00:00, 40.32it/s, Loss=0.0189763]

Local Optimization 31/175(w=19): 100%|██████████| 200/200 [00:07<00:00, 25.60it/s, Loss=0.0414702]

Global Optimization 31/175: 100%|██████████| 500/500 [00:21<00:00, 23.77it/s, Loss=0.